# 10.6 Feature Engineering

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.06_feature-engineering.ipynb)

Companion code for Section 10.6 of the book. Run the cells in order from top to bottom.

**Example 10.6.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import warnings; warnings.filterwarnings('ignore')

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print('=== Feature Engineering Impact on California Housing ===')

# ── Baseline: raw features ────────────────────────────────────────
X_raw = df.drop('MedHouseVal', axis=1)
y     = df['MedHouseVal']

model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                    max_depth=4, random_state=42)
scores_raw = cross_val_score(model, X_raw, y, cv=5, scoring='r2', n_jobs=-1)
print(f'\nBaseline (raw features):       R² = {scores_raw.mean():.4f} ± {scores_raw.std():.4f}')

# ── Feature Engineering ───────────────────────────────────────────
df_eng = df.copy()

# 1. Ratio features (more meaningful than raw totals)
df_eng['RoomsPerPerson']   = df_eng['AveRooms'] / (df_eng['AveOccup'] + 1e-6)
df_eng['BedroomRatio']     = df_eng['AveBedrms'] / (df_eng['AveRooms'] + 1e-6)
df_eng['PopDensity']       = df_eng['Population'] / (df_eng['AveOccup'] + 1e-6)

# 2. Log transforms for skewed features
df_eng['log_MedInc']       = np.log1p(df_eng['MedInc'])
df_eng['log_Population']   = np.log1p(df_eng['Population'])

# 3. Geographic features (distance from major cities)
cities = {'SF': (37.77, -122.42), 'LA': (34.05, -118.24), 'SD': (32.72, -117.16)}
for city, (lat, lon) in cities.items():
    df_eng[f'dist_{city}'] = np.sqrt(
        (df_eng['Latitude']-lat)**2 + (df_eng['Longitude']-lon)**2
    )

# 4. Interaction terms
df_eng['Income_x_Rooms']   = df_eng['MedInc'] * df_eng['AveRooms']
df_eng['HouseAge_x_Income']= df_eng['HouseAge'] * df_eng['MedInc']

# 5. Non-linear transforms
df_eng['MedInc_sq']        = df_eng['MedInc']**2
df_eng['MedInc_sqrt']      = np.sqrt(df_eng['MedInc'])

# 6. Binned features
df_eng['IncomeTier']  = pd.qcut(df_eng['MedInc'], q=5, labels=[0,1,2,3,4]).astype(int)
df_eng['AgeTier']     = pd.qcut(df_eng['HouseAge'], q=4, labels=[0,1,2,3]).astype(int)

X_eng = df_eng.drop('MedHouseVal', axis=1)
scores_eng = cross_val_score(model, X_eng, y, cv=5, scoring='r2', n_jobs=-1)
print(f'Engineered features:           R² = {scores_eng.mean():.4f} ± {scores_eng.std():.4f}')
print(f'Improvement:                   +{(scores_eng.mean()-scores_raw.mean())*100:.2f}% R²')
print(f'Features: {X_raw.shape[1]} raw → {X_eng.shape[1]} engineered')

# ── Feature importance of engineered features ─────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(X_eng, y, test_size=0.2, random_state=42)
model.fit(X_tr, y_tr)
fi = pd.Series(model.feature_importances_, index=X_eng.columns)
fi = fi.sort_values(ascending=False)
print(f'\nTop 10 Feature Importances (after engineering):')
print(fi.head(10).round(4))

# ── Automated feature engineering with PolynomialFeatures ─────────
print('\nPolynomial feature expansion:')
for degree in [1, 2, 3]:
    pipe = Pipeline([
        ('poly',  PolynomialFeatures(degree=degree, interaction_only=False, include_bias=False)),
        ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
    ])
    # Only use first 4 features to keep expansion manageable
    X_small = X_raw.iloc[:, :4]
    scores = cross_val_score(pipe, X_small, y, cv=3, scoring='r2', n_jobs=-1)
    n_feats = PolynomialFeatures(degree=degree).fit_transform(X_small).shape[1]
    print(f'  Degree {degree}: {X_small.shape[1]} → {n_feats} features, R²={scores.mean():.4f}')

# ── Visualise ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
fi.head(15).sort_values().plot(kind='barh', ax=ax, color='#1976D2', edgecolor='white')
ax.set_xlabel('Feature Importance (GradBoost)')
ax.set_title('Feature Importance after Engineering\n(engineered features in bold)',
            fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
# Highlight engineered features
raw_features = list(X_raw.columns)
for label in ax.get_yticklabels():
    if label.get_text() not in raw_features:
        label.set_fontweight('bold')
        label.set_color('#E53935')
plt.tight_layout()
plt.savefig('feature_engineering.png', dpi=150, bbox_inches='tight')
plt.show()